## Imports

In [5]:
import numpy as np
import pandas as pd

import ast
import re

## Load the data

In [2]:
data = pd.read_csv('../data/row/apartments_without_duplicates.csv')
data.head()

,id,price,address,coordinates,region,subway,rooms,footage,floor,features,residential,neighborhood,description,detail,attributes
0,11222088,1800,"Саперне Поле вул., 5а","30.53223038,50.41953659","Київ,Печерський р-н",Палац Україна,4 кімнати,150 / 130 / 20 м²,поверх 10 з 14,"['Бетонно монолітний', 'Роздільне', 'Перша зда...",ЖК Бульвар Фонтанів,['КНУКіМ'],Без животных\n\nЖК «Бульвар фонтанов» Саперное...,"Будинок - Бетонно монолітний, в квартирі 4 кім...","['Камін', 'Посудомийна машина', 'Душова кабіна..."
1,11222792,1000,"Саксаганського вул., 37к","30.51220894,50.43501282","Київ,Голосіївський р-н",Олімпійська,2 кімнати,67 / 57 / 10 м²,поверх 9 з 31,"['Бетонно монолітний', 'Кухня-вітальня', 'Перш...",ЖК Royal Tower,['ТЦ Олімпійський'],Без животных\n\nБЕЗ КОМИССИИ % !\n\nЕсть полны...,"Будинок - Бетонно монолітний, в квартирі 2 кім...","['Камін', 'Посудомийна машина', 'Душова кабіна..."
2,11295350,2200,"Костьольна вул., 4","30.52362633,50.45163727","Київ,Шевченківський р-н",Майдан Незалежності,4 кімнати,175 / 140 / 30 м²,поверх 2 з 5,"['Дореволюційний', 'Роздільне', 'Євроремонт']",NaN,"['Майдан Незалежності', 'Європейська площа', '...",Простора чотирикімнатна квартира доступна для ...,"Будинок - Дореволюційний, в квартирі 4 кімнати...","['Камін', 'Посудомийна машина', 'Душова кабіна..."
3,11292224,900,"Берестейський просп. (Перемоги), 11","30.48078156,50.44769287","Київ,Шевченківський р-н",Вокзальна,2 кімнати,55 / 25 / 20 м²,поверх 26 з 36,['Перша здача'],ЖК Manhattan city,"['Центральний РАЦС', 'Універмаг Україна', 'Пол...",— Апартаменти бізнес класу в центрі Києва \n— ...,В квартирі 2 кімнати. Вид з вікон на місто. Є ...,"['Посудомийна машина', 'Лічильники', 'Кондиціо..."
4,11278523,460,"Срібнокільська вул., 14","30.62314796,50.39974594","Київ,Дарницький р-н",Осокорки,3 кімнати,76 / 60 / 8 м²,поверх 3 з 16,"['Українська панель', 'Роздільне', 'Чудовий ст...",NaN,['Осокорки'],Здам 3-и кімнатну квартиру м.Осокорки вул. Срі...,"Будинок - Українська панель, в квартирі 3 кімн...","['Лічильники', 'Пральна машина', 'Ліжко', 'Хол..."


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11436 entries, 0 to 11435
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            11436 non-null  int64 
 1   price         11436 non-null  int64 
 2   address       11436 non-null  object
 3   coordinates   11307 non-null  object
 4   region        11436 non-null  object
 5   subway        9249 non-null   object
 6   rooms         11436 non-null  object
 7   footage       11436 non-null  object
 8   floor         11436 non-null  object
 9   features      11436 non-null  object
 10  residential   5502 non-null   object
 11  neighborhood  10348 non-null  object
 12  description   11436 non-null  object
 13  detail        11435 non-null  object
 14  attributes    6865 non-null   object
dtypes: int64(2), object(13)
memory usage: 1.3+ MB


## `price`

In [15]:
data['price'].describe()

count    1.143600e+04
mean     1.099805e+03
std      1.135821e+04
min      5.000000e+01
25%      3.800000e+02
50%      6.200000e+02
75%      1.100000e+03
max      1.150000e+06
Name: price, dtype: float64

`Price` is already in numeric format. But we need to check for outliers.

## `footage`

In [16]:
data['footage'].value_counts(dropna=False)

footage
50 / 30 / 10 м²          38
40 / 20 / 10 м²          36
45 / 30 / 10 м²          30
43 / 18 / 10 м²          25
55 / 30 / 15 м²          24
                         ..
45 / 15 / 23 м²           1
58.6 / 15.3 / 23.8 м²     1
42 / 11 / 20 м²           1
79.1 / 47.1 / 9 м²        1
127 / 64 / 18 м²          1
Name: count, Length: 6912, dtype: int64

Convert column 'footage' into three features: 'full_area', 'living_area', 'kitchen_area'. 

In [4]:
data[['full_area',
      'living_area',
      'kitchen_area']
] = data['footage'].str.replace(' м²', '').str.split(' / ', expand=True).astype('float')

In [18]:
data[['full_area',
      'living_area',
      'kitchen_area']].describe().round(2)

,full_area,living_area,kitchen_area
count,11436.00,11436.00,11436.00
mean,73.46,40.29,15.47
std,76.34,29.44,9.08
min,15.00,1.00,1.00
25%,45.00,20.00,10.00
50%,58.00,31.00,14.00
75%,86.00,50.00,20.00
max,6635.00,500.00,140.00


We need to check area-features for outliers.

In [9]:
data.loc[data['living_area'] == 1, ['price', 'footage', 'description']]

,price,footage,description
184,460,60 / 1 / 1 м²,Пропонується у довгострокову оренду 2-к кварти...
262,3200,240 / 1 / 1 м²,Предлагается в долгосрочную аренду БЕЗ КОМИССИ...
540,280,42 / 1 / 1 м²,"ул. В. Верховинца,10, 5/14 эт., 42/20/15. \nСв..."
570,1200,120 / 1 / 1 м²,Без комісії!\nПропонується в оренду 3-хкімнатн...
775,4200,122 / 1 / 1 м²,Предлагается в долгосрочную аренду 3-к кварти...
...,...,...,...
10538,800,70 / 1 / 1 м²,"Оренда квартири під офіс, з меблями, першій по..."
10608,370,37 / 1 / 1 м²,Сдам 1к квартиру с ремонтом в светлых тонах. Е...
10609,250,30 / 1 / 1 м²,вул. Иорданская (Гавро Лайоша) 24б метро Оболо...
11068,999,110 / 1 / 1 м²,"Без % Аренда 3к квартиры на Крещатике, с отдел..."


It seems like some apartments have missing information for the living and kitchen areas. We could label these apartments as "unknown_living_area" and replace values equal to `1` in the `living_area` and `kitchen_area` columns with the median value within groups defined by `rooms`, `full_area`, and `district`.

In [40]:
# Mark apartments with unknown living area
data['unknown_living_area'] = np.where(
    (data['living_area'] < 6) | (data['kitchen_area'] == 1), 1, 0
)

# Replace 1 with NaN for imputation
data['living_area'] = data['living_area'].mask(data['living_area'] < 6, np.nan)
data['kitchen_area'] = data['kitchen_area'].replace(1, np.nan)

# Fill remaining NaN with median by group
group_cols = ['rooms', 'full_area', 'district']

data['living_area'] = data['living_area'].fillna(
    data.groupby(group_cols)['living_area'].transform('median')
)

data['kitchen_area'] = data['kitchen_area'].fillna(
    data.groupby(group_cols)['kitchen_area'].transform('median')
)

In [41]:
data['living_area'].describe()

count    11432.000000
mean        40.595842
std         29.304556
min          7.000000
25%         20.000000
50%         32.000000
75%         50.000000
max        500.000000
Name: living_area, dtype: float64

## `rooms`

In [19]:
data['rooms'].value_counts(dropna=False)

rooms
2 кімнати             4059
1 кімната             4017
3 кімнати             2472
4 кімнати              678
5 кімнат               162
6 кімнат і більше       42
Вільне планування        4
2  кімн. в 2 кімн.       2
Name: count, dtype: int64

In [ ]:
data[data['rooms'] == '2  кімн. в 2 кімн.']

,id,price,address,coordinates,region,subway,rooms,footage,floor,features,residential,neighborhood,description,detail,attributes,full_area,living_area,kitchen_area
1454,11336743,800,"Завальна вул., 10г","30.6155014,50.39250183","Київ,Дарницький р-н",Осокорки,2 кімн. в 2 кімн.,79 / 38 / 18 м²,поверх 7 з 25,"['Бетонно монолітний', 'Суміжна', 'Дизайнерськ...",NaN,NaN,Вашій увазі пропонується двохкімнатна квартира...,"Будинок - Бетонно монолітний, в квартирі 2 кім...","['Посудомийна машина', 'Лічильники', 'Кондиціо...",79.0,38.0,18.0
4813,11132399,999,"Берестейський просп. (Перемоги), 42","30.45206833,50.45471573","Київ,Шевченківський р-н",Шулявська,2 кімн. в 2 кімн.,48 / 30 / 10 м²,поверх 3 з 10,"['Бетонно монолітний', 'Суміжно-роздільна', 'Д...",ЖК Crystal Park,"['Парк ім. Пушкіна', 'Шулявка', 'КПІ', 'Зоопарк']",Оренда просторої 1-2кімнатної квартири з якісн...,"Будинок - Бетонно монолітний, в квартирі 2 кім...","['Посудомийна машина', 'Душова кабіна', 'Лічил...",48.0,30.0,10.0


In [21]:
data[data['rooms'] == 'Вільне планування']

,id,price,address,coordinates,region,subway,rooms,footage,floor,features,residential,neighborhood,description,detail,attributes,full_area,living_area,kitchen_area
4785,11280048,460,"просп. Любомира Гузара, 15","30.45153999,50.42229462","Київ,Солом'янський р-н",NaN,Вільне планування,30 / 22 / 8 м²,поверх 13 з 17,"['Студія', 'Дизайнерський ремонт']",NaN,"['Чоколівка', 'Пологовий будинок № 5', 'Аеропо...","Чудова, затишна квартира в новому комплексі ""С...",Квартира вільного планування. Планування кімна...,"['Лічильники', 'Кондиціонер', 'Пральна машина'...",30.0,22.0,8.0
6541,11407113,1200,"Трьохсвятительська вул., 13","30.52193832,50.45497513","Київ,Шевченківський р-н",Поштова площа,Вільне планування,90 / 86 / 4 м²,поверх 3 з 4,"['Багаторівнева', 'Дизайнерський ремонт']",NaN,"['Михайлівськая площа', 'Володимирська Гірка']","За домовленістю, можливе встановлення електрич...",Квартира вільного планування. Планування кімна...,"['Камін', 'Сейф', 'Лічильники', 'Посуд', 'Ліжк...",90.0,86.0,4.0
7510,11426890,750,"Хорива вул., 15","30.51430702,50.46568298","Київ,Подільський р-н",Контрактова площа,Вільне планування,47 / 20 / 10 м²,поверх 2 з 4,"['Дореволюційний', 'Студія', 'Дизайнерський ре...",NaN,"['Контрактова площа', 'Житній ринок', 'Поділ',...",Здається стильна однокімнатна квартира в лофт-...,"Будинок - Дореволюційний, квартира вільного пл...",NaN,47.0,20.0,10.0
8840,11397310,1300,"Велика Васильківська вул. (Червоноармійська), 27","30.51645851,50.43849182","Київ,Печерський р-н",Площа Українських Героїв,Вільне планування,85 / 42 / 20 м²,поверх 3 з 7,"['Дореволюційний', 'Багаторівнева', 'Дизайнерс...",NaN,"['Спортивна площа', 'Парк ім. Шевченка', 'Бесс...","!!! Світло не вимикають, Газ!!!\nможна з твари...","Будинок - Дореволюційний, квартира вільного пл...","['Посудомийна машина', 'Душова кабіна', 'Лічил...",85.0,42.0,20.0


In [22]:
data.loc[data['rooms'] == 'Вільне планування', 'rooms'] = '1 кімната'

In [23]:
data['rooms'].value_counts()

rooms
2 кімнати             4059
1 кімната             4021
3 кімнати             2472
4 кімнати              678
5 кімнат               162
6 кімнат і більше       42
2  кімн. в 2 кімн.       2
Name: count, dtype: int64

Convert to number by deleting words

In [24]:
data['rooms'] = data['rooms'].str.split(' ').str[0].astype('int8')

In [25]:
data['rooms'].value_counts()

rooms
2    4061
1    4021
3    2472
4     678
5     162
6      42
Name: count, dtype: int64

## `floor`

Convert column 'floor' into two features: 'floor', 'num_storeys'. 

In [26]:
data['floor'].value_counts(dropna=False)

floor
поверх 5 з 5      219
поверх 4 з 5      214
поверх 3 з 5      206
поверх 7 з 9      195
поверх 2 з 5      192
                 ... 
поверх 9 з 33       1
поверх 31 з 34      1
поверх 3 з 35       1
поверх 29 з 31      1
поверх 13 з 34      1
Name: count, Length: 585, dtype: int64

In [27]:
data[['floor', 'num_storeys']] = (data['floor']
                                  .str.replace('поверх ', '')
                                  .str.split(' з ', expand=True)).astype('int')
data[['floor', 'num_storeys']]

,floor,num_storeys
0,10,14
1,9,31
2,2,5
3,26,36
4,3,16
...,...,...
11431,4,8
11432,3,3
11433,19,27
11434,2,4


In [29]:
data[['floor', 'num_storeys']].describe().round(2)

,floor,num_storeys
count,11436.00,11436.00
mean,9.67,17.06
std,6.63,8.34
min,1.00,1.00
25%,4.00,9.00
50%,8.00,17.00
75%,14.00,25.00
max,36.00,47.00


## `region`

In [31]:
data['region'].value_counts(dropna=False)

region
Київ,Печерський р-н        2579
Київ,Шевченківський р-н    1949
Київ,Голосіївський р-н     1507
Київ,Дарницький р-н        1411
Київ,Солом'янський р-н      922
Київ,Дніпровський р-н       834
Київ,Подільський р-н        699
Київ,Оболонський р-н        643
Київ,Святошинський р-н      530
Київ,Деснянський р-н        362
Name: count, dtype: int64

In [32]:
data['district'] = data['region'].str.split(',').str[1].str.split().str[0]
data['district'].value_counts(dropna=False)

district
Печерський        2579
Шевченківський    1949
Голосіївський     1507
Дарницький        1411
Солом'янський      922
Дніпровський       834
Подільський        699
Оболонський        643
Святошинський      530
Деснянський        362
Name: count, dtype: int64

## `subway`

In [33]:
data['subway'].value_counts(dropna=False)

subway
NaN                         2187
Звіринецька                  684
Вокзальна                    453
Либідська                    449
Лук'янівська                 445
Палац Україна                425
Золоті Ворота                343
Печерська                    339
Позняки                      337
Осокорки                     315
Лівобережна                  302
Олімпійська                  286
Васильківська                242
Університет                  227
Деміївська                   210
Кловська                     205
Контрактова площа            193
Нивки                        190
Славутич                     185
Шулявська                    184
Політехнічний інститут       176
Харківська                   176
Оболонь                      163
Виставковий центр            162
Хрещатик                     157
Сирець                       155
Академмістечко               152
Видубичі                     147
Мінська                      144
Голосіївська                 138
Іпо

## `coordiantes`

In [34]:
data['coordinates'].value_counts(dropna=False)

coordinates
NaN                        129
30.53064728,50.41595459     82
30.59981918,50.39749146     54
30.47374153,50.3926506      51
30.46647263,50.39138031     47
                          ... 
30.37978745,50.42261887      1
30.63508224,50.41643906      1
30.40856743,50.50588608      1
30.4904747,50.45685959       1
30.51169968,50.46016312      1
Name: count, Length: 5037, dtype: int64

In [35]:
data[['lon', 'lat']] = (data['coordinates']
                        .str.split(',', expand=True)
                        .apply(pd.to_numeric))

In [40]:
data[['lat', 'lon']].describe().round(3)

,lat,lon
count,11307.000,11307.000
mean,50.439,30.520
std,0.036,0.070
min,50.300,30.279
25%,50.413,30.477
50%,50.436,30.518
75%,50.458,30.550
max,50.568,30.738


## `neighborhood`

In [25]:
def safe_literal_eval(val):
    if pd.isna(val):
        return []
    try:
        return ast.literal_eval(val)
    except:
        return []

data['neighborhood'] = data['neighborhood'].apply(safe_literal_eval)
neighborhood_df = data['neighborhood'].apply(pd.Series)
neighborhood_df.columns = [f'neighborhood_{i+1}' for i in neighborhood_df.columns]
neighborhood_df.head()

,neighborhood_1,neighborhood_2,neighborhood_3,neighborhood_4,neighborhood_5
0,КНУКіМ,NaN,NaN,NaN,NaN
1,ТЦ Олімпійський,NaN,NaN,NaN,NaN
2,Майдан Незалежності,Європейська площа,Володимирська Гірка,NaN,NaN
3,Центральний РАЦС,Універмаг Україна,Політехнічний інститут,Ботанічний сад ім. акад. О. В. Фоміна,Парк ім. Шевченка
4,Осокорки,NaN,NaN,NaN,NaN


In [29]:
data = pd.concat([data, neighborhood_df], axis=1)

In [30]:
data.sample(5)

,id,price,address,coordinates,region,subway,rooms,footage,floor,features,...,kitchen_area,num_storeys,district,lon,lat,neighborhood_1,neighborhood_2,neighborhood_3,neighborhood_4,neighborhood_5
10854,11433191,620,"Зарічна вул., 2к3","30.60555649,50.39503098","Київ,Дарницький р-н",Славутич,2,60 / 35 / 10.3 м²,5,"['Бетонно монолітний', 'Роздільне', 'Чудовий с...",...,10.3,12,Дарницький,30.605556,50.395031,NaN,NaN,NaN,NaN,NaN
7602,11399315,700,"Максимовича вул. (Трутенка Онуфрія), 26г","30.47374153,50.3926506","Київ,Голосіївський р-н",Васильківська,1,41 / 14 / 18 м²,16,"['Українська цегла', 'Роздільне', 'Євроремонт']",...,18.0,20,Голосіївський,30.473742,50.392651,NaN,NaN,NaN,NaN,NaN
4332,11408278,470,Князя Романа Мстиславича вул. (Жмаченка генера...,"30.61091423,50.47012329","Київ,Дніпровський р-н",Дарниця,1,48.3 / 20 / 13 м²,20,"['Бетонно монолітний', 'Роздільне', 'Дизайнерс...",...,13.0,25,Дніпровський,30.610914,50.470123,Лікарня Швидкої Допомоги,Парк Перемога,NaN,NaN,NaN
3422,11402573,490,"Берестейський просп. (Перемоги), 11","30.47993469,50.4480629","Київ,Шевченківський р-н",Вокзальна,1,31 / 16 / 7 м²,30,"['Бетонно монолітний', 'Студія', 'Дизайнерськи...",...,7.0,36,Шевченківський,30.479935,50.448063,Центральний РАЦС,Охматдит,NaN,NaN,NaN
5573,11368427,370,"Підвисоцького професора вул., 20","30.55462837,50.41506577","Київ,Печерський р-н",Звіринецька,2,45 / 30 / 6 м²,4,"['Українська цегла', 'Роздільне', 'Хороший стан']",...,6.0,5,Печерський,30.554628,50.415066,Звіринець,Національний ботанічний сад ім. М. М. Гришка,Новопечерські липки,NaN,NaN


In [42]:
data.to_csv('../data/processed/apartments_cleaned.csv', index=False)